<a href="https://colab.research.google.com/github/PhilSocialScienceCouncil/PCRN-2026/blob/main/PCRNexcelprocessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Use this step-by-step notebook to update the resources available on the PCRN Archive webpage**

> Upload the updated excel spreadsheet below




In [13]:
#The upload file button will appear after this cell is run

import pandas as pd
import re
import os

from google.colab import files
uploaded=files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  filename = fn

Saving tentative google sheet submission.xlsx to tentative google sheet submission (4).xlsx
User uploaded file "tentative google sheet submission (4).xlsx" with length 10808 bytes


In [15]:
if 'uploaded' in locals() and uploaded:
    filename = next(iter(uploaded.keys())) # Get the first uploaded filename
else:
    raise FileNotFoundError("No file has been uploaded.")

# Create a new dataframe (basic spreadsheet style) from all of the separate sheets in this spreadsheet
all_sheets = pd.read_excel(filename, sheet_name=None, header=None) # Read without a header initially
dfs_with_journal = []
for sheet_name, sheet_df in all_sheets.items():
    if sheet_df.empty or sheet_df.shape[0] < 1:
        print(f"Skipping empty or short sheet: {sheet_name}")
        continue

    # Use the first row as the header and clean up column names
    header_row = sheet_df.iloc[0]
    header_row = header_row.astype(str).str.replace('nan', '').str.strip()

    sheet_df = sheet_df[1:].copy() # Data starts from the 2nd row (index 1)

    # Handle duplicate column names in the header
    cols = pd.Series(header_row)
    for dup in cols[cols.duplicated()].unique():
        cols[cols[cols == dup].index.values.tolist()] = [dup + '.' + str(i) if i != 0 else dup for i in range(len(cols[cols == dup].index.values.tolist()))]
    sheet_df.columns = cols

    # Print column names and their types for debugging
    print(f"Sheet: {sheet_name}")
    print("Columns and dtypes before concatenation:")
    print(sheet_df.columns)
    print(sheet_df.dtypes)
    print("-" * 30)

    dfs_with_journal.append(sheet_df)

# Concatenate all dataframes with journal information
df = pd.concat(dfs_with_journal, ignore_index=True)

# Select only the desired columns and exclude those starting with '.'
desired_columns = ['Timestamp', 'EntryID', 'theme', 'year', 'author(s)', 'institution(s)', 'publication_year', 'entry_type', 'age_group_target', 'geography', 'abstract', 'language', 'full_text_url', 'submitted_by', 'date_submitted', 'email_address']
# Ensure all desired columns exist before selecting
desired_columns_existing = [col for col in desired_columns if col in df.columns]
df = df[desired_columns_existing]


# Fill missing values with empty strings
df = df.fillna('')


# Function to extract year, handling errors and various formats, including multiple years
def extract_year(date_str):
    if pd.isna(date_str) or date_str == '':
        return []  # Return empty list for missing or empty values
    try:
        # Convert to string and find all occurrences of four-digit numbers
        date_str = str(date_str).strip()
        year_matches = re.findall(r'\\d{4}', date_str)

        extracted_years = []
        for year_str in year_matches:
            try:
                extracted_years.append(year_str) # Keep as string
            except ValueError:
                # Handle cases where a four-digit string is not a valid integer (shouldn't happen with \d{4} but as a safeguard)
                print(f"Warning: Could not convert '{year_str}' to integer in year string '{date_str}'")
                pass # Skip this invalid year

        return extracted_years if extracted_years else [] # Return the list of years or an empty list if none found

    except:
        # If any other error occurs, return an empty list
        return []


# Apply the function to the 'year' column
if 'year' in df.columns:
    df['year'] = df['year'].apply(extract_year)


# Define the columns to check for emptiness, excluding 'journal' and 'year'
columns_to_check_for_meaningful_text = [col for col in df.columns if col not in ['journal', 'year']]

# Remove rows where all specified columns (excluding 'journal' and 'year') are empty strings
df = df[~df[columns_to_check_for_meaningful_text].eq('').all(axis=1)]

df

Sheet: Form Responses 1
Columns and dtypes before concatenation:
Index(['Timestamp', 'EntryID', 'theme', 'year', 'author(s)', 'institution(s)',
       'publication_year', 'entry_type', 'age_group_target', 'geography',
       'abstract', 'language', 'full_text_url', 'full_text_available?',
       'submitted_by', 'date_submitted', 'email_address', '', 'Status',
       'Notes'],
      dtype='object', name=0)
0
Timestamp                object
EntryID                  object
theme                    object
year                     object
author(s)                object
institution(s)           object
publication_year         object
entry_type               object
age_group_target         object
geography                object
abstract                 object
language                 object
full_text_url            object
full_text_available?     object
submitted_by             object
date_submitted           object
email_address            object
                        float64
Status       

/tmp/ipykernel_3842/1474138614.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna('')


,Timestamp,EntryID,theme,year,author(s),institution(s),publication_year,entry_type,age_group_target,geography,abstract,language,full_text_url,submitted_by,date_submitted,email_address
0,2026-06-01 14:26:28.995,,Poverty/Social Protections,[],test,test,2026,White Paper,Middle childhood,test,test,English,google.com,test,2026-06-01,cmswartz@wm.edu
1,2026-06-01 14:27:14.344,,Education,[],test 1,test 1,2026,Policy Brief,Early Childhood,test 1,test 1,English,google.com,test 1,2026-06-01,cmswartz@wm.edu
2,2026-06-01 14:27:49.157,,Protection,[],test 2,test 2,2026,Journal Article,Early Childhood,test 2,test 3,English,google.com,test 2,2026-06-01,cmswartz@wm.edu
3,2026-06-01 14:42:03.419,ID-MV9J6G,Education,[],adsf,asdf,2026,Journal Article,Early Childhood,asdf,asdf,English,google.com,cam,2026-06-01,cmswartz@wm.edu
4,2026-06-01 14:42:36.314,ID-SGITT9,Poverty/Social Protections,[],asdf,asdf,2026,Working Paper,Early Childhood,asdf,asdf,English,google.com,asdf,2026-06-01,cmswartz@wm.edu


In [16]:
# Function to extract year, handling errors and various formats, including multiple years
def extract_year(date_str):
    # Check if the input is a list and if it's empty or contains only None/empty strings
    if isinstance(date_str, list):
        if not date_str or all(pd.isna(x) or x == '' for x in date_str):
            return [] # Return empty list for empty or all-empty lists
    # Handle non-list inputs (like individual NaNs or empty strings that might still come through)
    elif pd.isna(date_str) or date_str == '':
        return []  # Return empty list for missing or empty values

    try:
        # Convert to string and handle potential year ranges like "1982 & 1983" or "1982 and 1983"
        date_str = str(date_str).strip()
        # Find all occurrences of four-digit numbers
        year_matches = re.findall(r'\d{4}', date_str)

        extracted_years = []
        for year_str in year_matches:
            try:
                # Append the year as a string
                extracted_years.append(year_str)
            except ValueError:
                # Handle cases where a four-digit string is not a valid integer (shouldn't happen with \d{4} but as a safeguard)
                print(f"Warning: Could not convert '{year_str}' to integer in year string '{date_str}'")
                pass # Skip this invalid year

        return extracted_years if extracted_years else [] # Return the list of years or an empty list if none found

    except:
        # If any other error occurs, return an empty list
        return []

# Apply the function to the 'year' column
if 'year' in df.columns:
    # Ensure the column is treated as objects before applying the function that returns lists
    df['year'] = df['year'].astype(object).apply(extract_year)

**These cells convert the pdfs to images and handle any issues**

In [17]:
# --- Convert DataFrame to JSON ---
json_output_string = df.to_json(orient='records', indent=2)

In [18]:
import os

# --- Save JSON File ---
output_json_dir = "output_json"
output_json_path = os.path.join(output_json_dir, "PCRNdata.json")  # ← Change filename
os.makedirs(output_json_dir, exist_ok=True)

with open(output_json_path, 'w', encoding='utf-8') as f:
    f.write(json_output_string)

print(f"\nJSON file successfully created at: {output_json_path}")


JSON file successfully created at: output_json/PCRNdata.json


## Run these last two cells to download the new .json file and folder with images

In [20]:

from google.colab import files
files.download('output_json/PCRNdata.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>